In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# Load Output CSV
llm_df = pd.read_csv("results/llm_outputs.csv")
llm_df.head()

In [ ]:
# Check for null values
llm_df.info()

In [ ]:
# Drop Null Columns (They were not evaluated)
llm_df.drop('raw_response', axis=1, inplace=True)

In [ ]:
llm_df.dropna(inplace=True)
llm_df.info()

In [ ]:
numeric_cols = llm_df.select_dtypes(include= np.number).columns.to_list()
object_cols = llm_df.select_dtypes(include=['object']).columns.to_list()

In [ ]:
# Get the metrics
llm_df.describe()

In [ ]:
races = list(llm_df.race_group.unique())
races

fig, axes = plt.subplots(2, 2, figsize=(8, 10))
axes = axes.flatten()

for i, race in enumerate(races):
    axes[i].hist(llm_df[llm_df['race_group'] == race]['score'], edgecolor='black')
    axes[i].set_title(race)
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 6))

# Overall score distribution
axes[0].hist(llm_df['score'], edgecolor='black')
axes[0].set_title('Overall Score Distribution')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequency')

# Score distribution across jobs
jobs = llm_df['race_group'].unique()
for job in jobs:
    axes[1].hist(llm_df[llm_df['race_group'] == job]['score'], alpha=0.6, edgecolor='black', label=job)

axes[1].set_title('Score Distribution by Job')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=llm_df,
    x='race_group',
    y='score',
    palette='Set2',
    linewidth=1.5
)
plt.title('Score Distribution by Race Group', fontsize=14, fontweight='bold')
plt.xlabel('Race Group', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
output_df = llm_df[[
    'job_title_id',
    'race_group',
    'score',
    'rationale',
    'name_id',
    'name',
    'job_title',
    'prompt'
]]

race_dfs = {}

for race in output_df['race_group'].unique():
    race_dfs[race] = output_df[output_df['race_group'] == race][
        ['job_title', 'name', 'score', 'rationale']
    ].reset_index(drop=True)

In [ ]:
# Check the class distribution
for race, df in race_dfs.items():
    print(f"{race}: {len(df)} rows")

In [ ]:
races = list(race_dfs.keys())

comparison_df = race_dfs[races[0]][['job_title', 'score', 'rationale']].rename(
    columns={'score': f'score_{races[0]}', 'rationale': f'rationale_{races[0]}'}
)

for race in races[1:]:
    temp = race_dfs[race][['job_title', 'score', 'rationale']].rename(
        columns={'score': f'score_{race}', 'rationale': f'rationale_{race}'}
    )
    comparison_df = comparison_df.merge(temp, on=['job_title'], how='outer')

# Add prompt back from original data
prompt_map = output_df[['job_title', 'prompt']].drop_duplicates()
comparison_df = comparison_df.merge(prompt_map, on=['job_title'], how='left')

print(f"Total rows: {len(comparison_df)}")
comparison_df

In [ ]:
# Get score columns
score_cols = [col for col in comparison_df.columns if col.startswith('score_')]

# Flag rows where not all scores are equal
comparison_df['scores_vary'] = comparison_df[score_cols].nunique(axis=1) > 1
comparison_df['max_diff'] = comparison_df[score_cols].max(axis=1) - comparison_df[score_cols].min(axis=1)

varying_df = comparison_df[comparison_df['scores_vary']].sort_values('max_diff', ascending=False)

print(f"Rows with varying scores: {len(varying_df)} out of {len(comparison_df)}")
print(f"Max score difference: {varying_df['max_diff'].max()}")
print(f"Mean score difference: {varying_df['max_diff'].mean():.2f}")

varying_df.head(10)